# 03 — Train a 5-second JSBSim skill Transformer

This notebook consumes the canonical Parquet trajectories and label map produced by
`02_generate_jsbsim_skill_dataset.ipynb`. Complete flights are assigned to an
approximately **0.6:0.2:0.2 train:test:validation split**, stratified by the complete
set of tactical skills demonstrated in each flight.

Only `MODEL_FEATURE_COLUMNS` are model inputs. Commanded skills, native actions, and
other privileged columns are targets or audit metadata only. Transition-spanning
windows are excluded by default; set `BVR_KEEP_MIXED_WINDOWS=1` to label them by their
final sample.

The Transformer is trained on the training split. In accordance with this experiment's
selection protocol, test loss controls checkpointing and early stopping. The validation
split remains untouched until the selected checkpoint receives its final evaluation.
MLflow records configuration, per-epoch metrics, the checkpoint, and the complete
reproducibility bundle.

## 1. Imports, reproducibility, and MLflow configuration

The dataset path exactly matches notebook 02's default output. Environment variables
allow short, reproducible experiments without editing cells. CPU is the safe default;
set `BVR_TRAIN_DEVICE=cuda` only after verifying the local CUDA installation.

In [ ]:
from __future__ import annotations

import gc
import json
import math
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as pads
import torch
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from bvr_behavior_prediction.data.observable_columns import MODEL_FEATURE_COLUMNS
from bvr_behavior_prediction.data.privileged_columns import PRIVILEGED_COLUMNS

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent
DATASET_DIR = Path(os.getenv(
    "BVR_TRAIN_DATASET",
    REPO_ROOT / "artifacts/datasets/bvr_f16_1v1_jsbsim_skills_v001",
))
OUTPUT_DIR = Path(os.getenv(
    "BVR_CLASSIFIER_OUTPUT", REPO_ROOT / "artifacts/models/jsbsim_skill_transformer_v001"
))
MLFLOW_TRACKING_URI = os.getenv(
    "BVR_MLFLOW_TRACKING_URI", (REPO_ROOT / "artifacts/mlruns").resolve().as_uri()
)
MLFLOW_EXPERIMENT = os.getenv("BVR_MLFLOW_EXPERIMENT", "jsbsim-skill-transformer")
WINDOW_S = 5.0
STRIDE_S = float(os.getenv("BVR_WINDOW_STRIDE_S", "1.0"))
KEEP_MIXED_WINDOWS = os.getenv("BVR_KEEP_MIXED_WINDOWS", "0") == "1"
BATCH_SIZE = int(os.getenv("BVR_BATCH_SIZE", "64"))
DATALOADER_WORKERS = int(os.getenv("BVR_DATALOADER_WORKERS", "0"))
USE_AMP = os.getenv("BVR_MIXED_PRECISION", "1") == "1"
TORCH_THREADS = int(os.getenv("BVR_TORCH_THREADS", str(min(4, os.cpu_count() or 1))))
REQUESTED_DEVICE = os.getenv("BVR_TRAIN_DEVICE", "cpu").strip().lower()
EPOCHS = int(os.getenv("BVR_TRAIN_EPOCHS", "30"))
PATIENCE = int(os.getenv("BVR_EARLY_STOPPING_PATIENCE", "6"))
LEARNING_RATE = float(os.getenv("BVR_LEARNING_RATE", "0.001"))
D_MODEL = int(os.getenv("BVR_TRANSFORMER_D_MODEL", "128"))
NHEAD = int(os.getenv("BVR_TRANSFORMER_HEADS", "8"))
NUM_LAYERS = int(os.getenv("BVR_TRANSFORMER_LAYERS", "3"))
DROPOUT = float(os.getenv("BVR_TRANSFORMER_DROPOUT", "0.2"))
SEED = int(os.getenv("BVR_TRAIN_SEED", "20260911"))
SPLIT_FRACTIONS = {"train": 0.60, "test": 0.20, "validation": 0.20}

if BATCH_SIZE < 1 or DATALOADER_WORKERS < 0:
    raise ValueError("Batch size must be positive and data-loader workers cannot be negative")
if TORCH_THREADS < 1:
    raise ValueError("BVR_TORCH_THREADS must be at least 1")
if D_MODEL % NHEAD:
    raise ValueError("BVR_TRANSFORMER_D_MODEL must be divisible by BVR_TRANSFORMER_HEADS")
if REQUESTED_DEVICE.startswith("cuda") and not torch.cuda.is_available():
    raise RuntimeError(
        f"BVR_TRAIN_DEVICE={REQUESTED_DEVICE!r} requires CUDA, but this PyTorch "
        "installation cannot access it. Use BVR_TRAIN_DEVICE=cpu or install a "
        "CUDA-compatible PyTorch build."
    )

torch.set_num_threads(TORCH_THREADS)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if REQUESTED_DEVICE.startswith("cuda"):
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device(REQUESTED_DEVICE)
AMP_ENABLED = USE_AMP and DEVICE.type == "cuda"
if DEVICE.type == "cuda":
    # TF32 speeds supported matrix multiplications without increasing memory use.
    torch.set_float32_matmul_precision("high")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)
print({"dataset": str(DATASET_DIR), "device": str(DEVICE), "seed": SEED,
       "mlflow_tracking_uri": mlflow.get_tracking_uri()})

## 2. Load notebook 02's canonical dataset efficiently

Only columns needed by this notebook are scanned from Parquet. PyArrow reads the shards
as one dataset and transfers columns to pandas without a second consolidated copy. Model
features are immediately narrowed to `float32`, substantially reducing host-memory use
before window construction.


In [ ]:
shards = sorted((DATASET_DIR / "trajectories").glob("*.parquet"))
label_map_path = DATASET_DIR / "label_map.json"
if not shards or not label_map_path.exists():
    raise FileNotFoundError(
        f"Expected trajectories/*.parquet and label_map.json under {DATASET_DIR}. "
        "Run notebooks/02_generate_jsbsim_skill_dataset.ipynb first or set "
        "BVR_TRAIN_DATASET."
    )

LABELS = json.loads(label_map_path.read_text())["tactical"]
label_to_index = {label: index for index, label in enumerate(LABELS)}
required_columns = ["episode_id", "time_s", "tactical_label", *MODEL_FEATURE_COLUMNS]
try:
    arrow_table = pads.dataset(shards, format="parquet").to_table(columns=required_columns)
    trajectories = arrow_table.to_pandas(split_blocks=True, self_destruct=True)
    del arrow_table
except pa.ArrowKeyError as error:
    raise RuntimeError(
        "PyArrow's legacy extension registry is incompatible with this pandas session. "
        "Install the project dependencies (which require pyarrow>=14.0.1), restart the "
        "notebook kernel, and run all cells again."
    ) from error

missing = set(required_columns).difference(trajectories.columns)
assert not missing, f"Missing required columns: {sorted(missing)}"
assert set(MODEL_FEATURE_COLUMNS).isdisjoint(PRIVILEGED_COLUMNS)
unknown = set(trajectories["tactical_label"].unique()).difference(label_to_index)
assert not unknown, f"Labels absent from label_map.json: {sorted(unknown)}"
trajectories = trajectories.astype(
    {column: np.float32 for column in MODEL_FEATURE_COLUMNS}, copy=False
)
trajectories.sort_values(["episode_id", "time_s"], inplace=True, ignore_index=True)
assert np.isfinite(trajectories.loc[:, MODEL_FEATURE_COLUMNS].to_numpy(copy=False)).all(), \
    "Observable features contain NaN or infinity"
print(f"Loaded {len(trajectories):,} samples from {trajectories.episode_id.nunique():,} episodes")
display(trajectories.groupby("tactical_label").agg(
    samples=("time_s", "size"), episodes=("episode_id", "nunique")
))


## 3. Stratify complete episodes 60:20:20 by demonstrated skills

A flight can demonstrate more than one skill. Its stratum is therefore the sorted set
of all tactical labels appearing in that episode, rather than only its first or most
frequent label. A 60/40 stratified split is followed by an equal stratified division of
the remainder. This preserves joint skill combinations while keeping every flight—and
all overlapping windows from it—in exactly one split.

Every demonstrated-skill combination needs at least five episodes to be represented in
all three partitions. The production dataset from notebook 02 readily satisfies this;
a deliberately tiny smoke dataset fails with an actionable message rather than silently
falling back to an unstratified split.

In [ ]:
episode_skills = (
    trajectories.groupby("episode_id")["tactical_label"]
    .agg(lambda values: tuple(sorted(set(values))))
    .rename("skills")
    .reset_index()
)
episode_skills["stratum"] = episode_skills["skills"].map("|".join)
stratum_counts = episode_skills["stratum"].value_counts()
if len(episode_skills) < 5 or (stratum_counts < 5).any():
    rare = stratum_counts[stratum_counts < 5].to_dict()
    raise ValueError(
        "Stratified 60:20:20 splitting requires at least five flights for every "
        f"demonstrated-skill combination; insufficient strata: {rare}. Generate more flights."
    )

train_episodes, selection_episodes = train_test_split(
    episode_skills,
    test_size=SPLIT_FRACTIONS["test"] + SPLIT_FRACTIONS["validation"],
    random_state=SEED,
    shuffle=True,
    stratify=episode_skills["stratum"],
)
test_episodes, validation_episodes = train_test_split(
    selection_episodes,
    test_size=0.5,
    random_state=SEED,
    shuffle=True,
    stratify=selection_episodes["stratum"],
)
split_tables = {
    "train": train_episodes,
    "test": test_episodes,
    "validation": validation_episodes,
}
split_ids = {name: table["episode_id"].tolist() for name, table in split_tables.items()}
all_split_ids = [set(ids) for ids in split_ids.values()]
assert all(all_split_ids) and not any(
    all_split_ids[i] & all_split_ids[j]
    for i in range(len(all_split_ids)) for j in range(i + 1, len(all_split_ids))
)
assert set().union(*all_split_ids) == set(episode_skills["episode_id"])

split_audit = pd.concat([
    table.assign(split=name).explode("skills")
    for name, table in split_tables.items()
])
split_summary = pd.crosstab(split_audit["skills"], split_audit["split"])
split_summary.loc["TOTAL EPISODES"] = {
    name: len(table) for name, table in split_tables.items()
}
print({name: round(len(ids) / len(episode_skills), 4) for name, ids in split_ids.items()})
display(split_summary)

## 4. Construct five-second windows without leakage

Cadence is inferred and checked across all episodes. Splitting happened first, so no
sample or overlapping window can cross a partition boundary. Window arrays are counted
and preallocated before they are filled; this avoids a large Python list of views and
the additional peak-memory copy previously required by `np.stack`.


In [ ]:
sample_deltas = trajectories.groupby("episode_id")["time_s"].diff().dropna()
SAMPLE_DT_S = float(sample_deltas.median())
assert SAMPLE_DT_S > 0 and np.allclose(sample_deltas, SAMPLE_DT_S, atol=1e-6), \
    "Irregular sample cadence"
WINDOW_SAMPLES = int(round(WINDOW_S / SAMPLE_DT_S))
STRIDE_SAMPLES = max(1, int(round(STRIDE_S / SAMPLE_DT_S)))
assert WINDOW_SAMPLES >= 2


def make_windows(frame, allowed_episode_ids):
    allowed_episode_ids = set(allowed_episode_ids)
    def selected_episodes():
        return (
            (episode_id, episode)
            for episode_id, episode in frame.groupby("episode_id", sort=False)
            if episode_id in allowed_episode_ids
        )

    window_count = mixed_count = 0
    for _, episode in selected_episodes():
        targets = episode["tactical_label"].to_numpy(copy=False)
        for start in range(0, len(episode) - WINDOW_SAMPLES + 1, STRIDE_SAMPLES):
            window_targets = targets[start:start + WINDOW_SAMPLES]
            mixed = bool(np.any(window_targets != window_targets[0]))
            mixed_count += int(mixed)
            window_count += int(KEEP_MIXED_WINDOWS or not mixed)
    if not window_count:
        raise ValueError("No windows were produced; check episode duration and filtering")

    sequences = np.empty(
        (window_count, WINDOW_SAMPLES, len(MODEL_FEATURE_COLUMNS)), dtype=np.float32
    )
    labels = np.empty(window_count, dtype=np.int64)
    metadata = []
    output_index = 0
    for episode_id, episode in selected_episodes():
        values = episode.loc[:, MODEL_FEATURE_COLUMNS].to_numpy(dtype=np.float32, copy=False)
        targets = episode["tactical_label"].to_numpy(copy=False)
        times = episode["time_s"].to_numpy(copy=False)
        for start in range(0, len(episode) - WINDOW_SAMPLES + 1, STRIDE_SAMPLES):
            stop = start + WINDOW_SAMPLES
            window_targets = targets[start:stop]
            mixed = bool(np.any(window_targets != window_targets[0]))
            if mixed and not KEEP_MIXED_WINDOWS:
                continue
            sequences[output_index] = values[start:stop]
            labels[output_index] = label_to_index[window_targets[-1]]
            metadata.append({
                "episode_id": episode_id,
                "start_time_s": float(times[start]),
                "end_time_s": float(times[stop - 1]),
                "mixed_skill": mixed,
            })
            output_index += 1
    return sequences, labels, pd.DataFrame.from_records(metadata), mixed_count


raw = {}
for split_name, ids in split_ids.items():
    x, y, metadata, mixed = make_windows(trajectories, ids)
    raw[split_name] = {"x": x, "y": y, "metadata": metadata}
    print(split_name, {"episodes": len(ids), "windows": len(y), "excluded_mixed": mixed})

for left, right in (("train", "test"), ("train", "validation"), ("test", "validation")):
    assert set(raw[left]["metadata"].episode_id).isdisjoint(raw[right]["metadata"].episode_id)

# The row-level frame is no longer needed. Release it before allocating loader buffers.
del trajectories, sample_deltas
gc.collect()


## 5. Fit preprocessing on training data only

The scaler sees only training timesteps. Every `float32` window array is then normalized
**in place**, rather than retaining both raw and scaled copies. Data loaders pin only
individual batches when CUDA is active, enabling asynchronous host-to-device copies
without pinning the full dataset.


In [ ]:
scaler = StandardScaler(copy=False)
train_shape = raw["train"]["x"].shape
scaler.fit(raw["train"]["x"].reshape(-1, train_shape[-1]))
normalization_mean = scaler.mean_.astype(np.float32)
normalization_scale = scaler.scale_.astype(np.float32)

loaders = {}
for split_name, values in raw.items():
    x = values["x"]
    x -= normalization_mean
    x /= normalization_scale
    dataset = TensorDataset(torch.from_numpy(x), torch.from_numpy(values["y"]))
    generator = torch.Generator().manual_seed(SEED)
    loaders[split_name] = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=split_name == "train",
        generator=generator if split_name == "train" else None,
        num_workers=DATALOADER_WORKERS,
        pin_memory=DEVICE.type == "cuda",
        persistent_workers=DATALOADER_WORKERS > 0,
    )
print("Train tensor:", tuple(loaders["train"].dataset.tensors[0].shape))


## 6. Memory-efficient Transformer training tracked by MLflow

CUDA training uses automatic mixed precision by default, reducing activation and gradient
memory while accelerating Tensor Core-capable GPUs. The fused CUDA AdamW implementation,
TF32 matrix multiplication, pinned batch transfers, and inference mode accelerate the hot
paths without placing the complete dataset on the GPU. Set `BVR_MIXED_PRECISION=0` for a
full-float32 diagnostic run, lower `BVR_BATCH_SIZE` if still memory constrained, and tune
`BVR_DATALOADER_WORKERS` to overlap host loading with GPU work.

After every epoch, **test loss** is evaluated. An improvement writes `checkpoint.pt`;
test loss also drives patience-based early stopping. MLflow receives parameters and
train/test metrics for every epoch.


In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_length, dropout):
        super().__init__()
        positions = torch.arange(max_length, dtype=torch.float32).unsqueeze(1)
        frequencies = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10_000.0) / d_model)
        )
        encoding = torch.zeros(max_length, d_model)
        encoding[:, 0::2] = torch.sin(positions * frequencies)
        encoding[:, 1::2] = torch.cos(positions * frequencies)
        self.register_buffer("encoding", encoding.unsqueeze(0), persistent=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, sequence):
        return self.dropout(sequence + self.encoding[:, :sequence.size(1)])


class SkillTransformer(nn.Module):
    def __init__(self, input_dim, class_count, window_samples, d_model=128,
                 nhead=8, layers=3, dropout=0.2):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, d_model)
        self.positions = SinusoidalPositionalEncoding(d_model, window_samples, dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=4 * d_model,
            dropout=dropout, activation="gelu", batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=layers, norm=nn.LayerNorm(d_model)
        )
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(d_model, class_count))

    def forward(self, sequence):
        encoded = self.encoder(self.positions(self.input_projection(sequence)))
        return self.classifier(encoded.mean(dim=1))


model_config = {
    "input_dim": len(MODEL_FEATURE_COLUMNS), "class_count": len(LABELS),
    "window_samples": WINDOW_SAMPLES, "d_model": D_MODEL, "nhead": NHEAD,
    "layers": NUM_LAYERS, "dropout": DROPOUT,
}
model = SkillTransformer(**model_config).to(DEVICE)
counts = np.bincount(raw["train"]["y"], minlength=len(LABELS))
weights = counts.sum() / (len(LABELS) * np.maximum(counts, 1))
criterion = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32, device=DEVICE))
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4, fused=DEVICE.type == "cuda"
)
grad_scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)


def run_epoch(loader, training=False):
    model.train(training)
    total_loss = total_correct = total = 0
    for features, target in loader:
        features = features.to(DEVICE, non_blocking=True)
        target = target.to(DEVICE, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training), torch.autocast(
            device_type=DEVICE.type, dtype=torch.float16, enabled=AMP_ENABLED
        ):
            logits = model(features)
            loss = criterion(logits, target)
            if training:
                grad_scaler.scale(loss).backward()
                grad_scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                grad_scaler.step(optimizer)
                grad_scaler.update()
        total_loss += loss.item() * len(target)
        total_correct += (logits.argmax(1) == target).sum().item()
        total += len(target)
    return {"loss": total_loss / total, "accuracy": total_correct / total}


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = OUTPUT_DIR / "checkpoint.pt"
history, best_test_loss, stale_epochs = [], float("inf"), 0
mlflow_params = {
    **model_config, "architecture": "transformer_encoder", "batch_size": BATCH_SIZE,
    "epochs": EPOCHS, "patience": PATIENCE, "learning_rate": LEARNING_RATE,
    "seed": SEED, "device": str(DEVICE), "mixed_precision": AMP_ENABLED,
    "dataloader_workers": DATALOADER_WORKERS, "window_s": WINDOW_S, "stride_s": STRIDE_S,
    **{f"split_{name}": fraction for name, fraction in SPLIT_FRACTIONS.items()},
}

with mlflow.start_run(run_name=f"transformer-seed-{SEED}") as active_run:
    run_id = active_run.info.run_id
    mlflow.log_params(mlflow_params)
    mlflow.set_tags({"dataset": DATASET_DIR.name, "checkpoint_selection_split": "test"})
    for epoch in range(1, EPOCHS + 1):
        train_metrics = run_epoch(loaders["train"], training=True)
        test_metrics = run_epoch(loaders["test"])
        row = {"epoch": epoch,
               **{f"train_{key}": value for key, value in train_metrics.items()},
               **{f"test_{key}": value for key, value in test_metrics.items()}}
        history.append(row)
        mlflow.log_metrics({key: value for key, value in row.items() if key != "epoch"}, step=epoch)
        print(f"{epoch:02d} train loss={train_metrics['loss']:.4f} "
              f"acc={train_metrics['accuracy']:.3f} test loss={test_metrics['loss']:.4f} "
              f"acc={test_metrics['accuracy']:.3f}")
        if test_metrics["loss"] < best_test_loss - 1e-4:
            best_test_loss = test_metrics["loss"]
            stale_epochs = 0
            torch.save({
                "epoch": epoch, "test_loss": best_test_loss,
                "model_state_dict": model.state_dict(), "model_config": model_config,
            }, CHECKPOINT_PATH)
        else:
            stale_epochs += 1
            if stale_epochs >= PATIENCE:
                print("Early stopping on test loss")
                break

    checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
    model.load_state_dict(checkpoint["model_state_dict"])
    mlflow.log_metric("best_test_loss", checkpoint["test_loss"])
    mlflow.log_metric("best_epoch", checkpoint["epoch"])
    mlflow.log_artifact(str(CHECKPOINT_PATH), artifact_path="checkpoints")

history = pd.DataFrame(history)
history.plot(x="epoch", y=["train_loss", "test_loss"], grid=True,
             title="MLflow-tracked learning curves")
plt.show()
print({"mlflow_run_id": run_id, "selected_epoch": checkpoint["epoch"]})

## 7. Final evaluation on the untouched validation set

Validation is not used for scaling, optimization, checkpoint selection, or early
stopping. It provides the final class-wise report for the test-selected checkpoint.

In [ ]:
def predict(loader):
    model.eval()
    actual, predicted = [], []
    with torch.inference_mode():
        for features, target in loader:
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP_ENABLED):
                logits = model(features.to(DEVICE, non_blocking=True))
            actual.extend(target.numpy().tolist())
            predicted.extend(logits.argmax(1).cpu().numpy().tolist())
    return np.asarray(actual), np.asarray(predicted)


y_validation, y_pred = predict(loaders["validation"])
validation_accuracy = float((y_validation == y_pred).mean())
report = classification_report(
    y_validation, y_pred, labels=np.arange(len(LABELS)), target_names=LABELS,
    zero_division=0, digits=3, output_dict=True,
)
print(classification_report(
    y_validation, y_pred, labels=np.arange(len(LABELS)), target_names=LABELS,
    zero_division=0, digits=3,
))
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(
    y_validation, y_pred, labels=np.arange(len(LABELS)), display_labels=LABELS,
    normalize="true", xticks_rotation=45, cmap="Blues", ax=ax,
)
ax.set_title("Untouched validation confusion matrix (row normalized)")
plt.tight_layout()
plt.show()

## 8. Save and log the reproducible bundle

The bundle includes the test-selected Transformer, training-only normalization,
stratified episode assignments, class and feature order, window configuration, history,
and validation report. The same files are attached to the active MLflow run.

In [ ]:
torch.save({
    "model_state_dict": model.state_dict(), "model_config": model_config,
    "feature_columns": list(MODEL_FEATURE_COLUMNS), "labels": LABELS,
    "window_samples": WINDOW_SAMPLES, "sample_dt_s": SAMPLE_DT_S,
    "selected_epoch": checkpoint["epoch"], "selection_test_loss": checkpoint["test_loss"],
}, OUTPUT_DIR / "model.pt")
(OUTPUT_DIR / "preprocessing.json").write_text(json.dumps({
    "feature_columns": list(MODEL_FEATURE_COLUMNS),
    "mean": scaler.mean_.tolist(), "scale": scaler.scale_.tolist(),
}, indent=2))
(OUTPUT_DIR / "split.json").write_text(json.dumps({
    "seed": SEED, "fractions": SPLIT_FRACTIONS, "stratification": "demonstrated_skill_set",
    "episode_ids": split_ids, "window_s": WINDOW_S, "stride_s": STRIDE_S,
    "keep_mixed_windows": KEEP_MIXED_WINDOWS,
}, indent=2))
history.to_csv(OUTPUT_DIR / "history.csv", index=False)
(OUTPUT_DIR / "validation_report.json").write_text(json.dumps(report, indent=2))
for split_name, values in raw.items():
    values["metadata"].assign(label=[LABELS[index] for index in values["y"]]).to_parquet(
        OUTPUT_DIR / f"{split_name}_windows.parquet", index=False
    )

with mlflow.start_run(run_id=run_id):
    mlflow.log_metric("validation_accuracy", validation_accuracy)
    mlflow.log_artifacts(str(OUTPUT_DIR), artifact_path="training_bundle")
print("Saved training bundle to", OUTPUT_DIR.resolve())

## Interpretation notes

* Results are episode-held-out, not independent random-window performance.
* The test partition is intentionally a **development selection set** in this protocol;
  report the untouched validation metrics as the final generalization estimate.
* Per-class metrics matter because aggregate accuracy can hide weak rare-skill behavior.
* Mixed windows are better suited to a future transition or multi-label model.
* Synthetic performance does not establish real-world tactical generalization.